### ETL Gold — Financial Transactions Dataset

This notebook reads the silver tables, enriches them, and creates aggregated gold tables
to answer the business questions defined in the project requirements.

#### Read Silver Tables

In [0]:
transactions = spark.read.table("jarvis_databricks.silver.transactions_data_silver")
cards = spark.read.table("jarvis_databricks.silver.cards_data_silver")
users = spark.read.table("jarvis_databricks.silver.users_data_silver")

display(transactions.limit(5))

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,transaction_year,transaction_month,transaction_day,day_of_week_num,day_of_week,hour_of_day,time_of_day,mcc_description,is_fraud
9121784,2011-02-06T17:35:00.000Z,1242,4162,-417.0000,Swipe Transaction,92822,Miami Beach,FL,33141.0,3395,null,2011,2,6,1,Sunday,17,Evening,Welding Repair,false
9274297,2011-03-14T16:59:00.000Z,1344,1027,-96.0000,Swipe Transaction,61195,Sterling Heights,MI,48313.0,5541,null,2011,3,14,2,Monday,16,Afternoon,Service Stations,false
9407732,2011-04-14T20:39:00.000Z,1340,2954,60.6200,Swipe Transaction,61195,Houston,TX,77064.0,5541,null,2011,4,14,5,Thursday,20,Evening,Service Stations,false
9545803,2011-05-17T09:31:00.000Z,1283,4570,67.0000,Swipe Transaction,59935,Camas,WA,98607.0,5499,null,2011,5,17,3,Tuesday,9,Morning,Miscellaneous Food Stores,false
9587092,2011-05-26T19:12:00.000Z,1198,2804,49.5500,Online Transaction,41122,ONLINE,null,null,4784,null,2011,5,26,5,Thursday,19,Evening,Tolls and Bridge Fees,false


#### Prepare Enriched Transactions (base for gold tables)

In [0]:
from pyspark.sql.functions import col, abs as spark_abs, date_trunc, to_date

transactions_enriched = (
    transactions
    .withColumn("abs_amount", spark_abs(col("amount")))
    .withColumn("week_start", to_date(date_trunc("week", col("date"))))
)

display(transactions_enriched.limit(5))

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,transaction_year,transaction_month,transaction_day,day_of_week_num,day_of_week,hour_of_day,time_of_day,mcc_description,is_fraud,abs_amount,week_start
9121784,2011-02-06T17:35:00.000Z,1242,4162,-417.0000,Swipe Transaction,92822,Miami Beach,FL,33141.0,3395,null,2011,2,6,1,Sunday,17,Evening,Welding Repair,false,417.0000,2011-01-31
9274297,2011-03-14T16:59:00.000Z,1344,1027,-96.0000,Swipe Transaction,61195,Sterling Heights,MI,48313.0,5541,null,2011,3,14,2,Monday,16,Afternoon,Service Stations,false,96.0000,2011-03-14
9407732,2011-04-14T20:39:00.000Z,1340,2954,60.6200,Swipe Transaction,61195,Houston,TX,77064.0,5541,null,2011,4,14,5,Thursday,20,Evening,Service Stations,false,60.6200,2011-04-11
9545803,2011-05-17T09:31:00.000Z,1283,4570,67.0000,Swipe Transaction,59935,Camas,WA,98607.0,5499,null,2011,5,17,3,Tuesday,9,Morning,Miscellaneous Food Stores,false,67.0000,2011-05-16
9587092,2011-05-26T19:12:00.000Z,1198,2804,49.5500,Online Transaction,41122,ONLINE,null,null,4784,null,2011,5,26,5,Thursday,19,Evening,Tolls and Bridge Fees,false,49.5500,2011-05-23


In [0]:
from pyspark.sql.functions import count, countDistinct, sum as spark_sum, avg, round as spark_round, when, coalesce, lit, col, to_date

fraud_daily_summary_gold = (
    transactions_enriched
    .withColumn("day", to_date(col("date")))
    .groupBy("day")
    .agg(
        count("*").alias("total_transaction_count"),
        countDistinct("client_id").alias("unique_users"),
        countDistinct("merchant_id").alias("unique_merchants"),
        spark_sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transaction_count"),
        countDistinct(when(col("is_fraud") == True, col("client_id"))).alias("unique_fraud_users"),
        spark_round(spark_sum("amount"), 2).alias("total_amount"),
        spark_round(spark_sum(when(col("is_fraud") == True, col("abs_amount")).otherwise(lit(0.0))), 2).alias("fraud_amount"),
        spark_round(spark_sum(when(col("is_fraud") == True, 1).otherwise(0)) / count("*"), 4).alias("fraud_rate"),
        coalesce(spark_round(avg(when(col("is_fraud") == True, col("abs_amount"))), 2), lit(0.0)).alias("avg_fraud_amount"),
        coalesce(spark_round(avg(when(col("is_fraud") == False, col("abs_amount"))), 2), lit(0.0)).alias("avg_non_fraud_amount")
    )
    .orderBy("day")
)

display(fraud_daily_summary_gold.limit(20))

day,total_transaction_count,unique_users,unique_merchants,fraud_transaction_count,unique_fraud_users,total_amount,fraud_amount,fraud_rate,avg_fraud_amount,avg_non_fraud_amount
2010-01-01,3463,978,1010,1,1,124498.32,0.19,3.0E-4,0.19,51.49
2010-01-02,2989,954,986,0,0,138700.62,0.0,0.0,0.0,54.46
2010-01-03,3311,983,1009,1,1,135016.77,339.0,3.0E-4,339.0,49.12
2010-01-04,3244,988,958,2,1,131315.75,11.64,6.0E-4,5.82,48.96
2010-01-05,3330,980,995,1,1,143760.66,8.76,3.0E-4,8.76,52.86
2010-01-06,3365,992,1034,0,0,139046.49,0.0,0.0,0.0,51.33
2010-01-07,3346,984,1014,2,1,150784.43,629.54,6.0E-4,314.77,55.45
2010-01-08,3016,973,994,4,4,142249.31,383.24,0.0013,95.81,54.94
2010-01-09,3102,956,1018,1,1,138220.61,23.1,3.0E-4,23.1,61.07
2010-01-10,3416,988,1049,5,3,150998.46,530.01,0.0015,106.0,53.83


In [0]:
from pyspark.sql.functions import count, sum as spark_sum, when, col, round as spark_round

fraud_by_time_of_day_gold = (
    transactions_enriched
    .groupBy("time_of_day")
    .agg(
        count("*").alias("total_transaction_count"),
        spark_sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transaction_count"),
        spark_round(spark_sum(when(col("is_fraud") == True, 1).otherwise(0)) / count("*"), 4).alias("fraud_rate")
    )
    .orderBy(col("fraud_rate").desc())
)

display(fraud_by_time_of_day_gold)

time_of_day,total_transaction_count,fraud_transaction_count,fraud_rate
Afternoon,4464677,5693,0.0013
Morning,5415684,5741,0.0011
Evening,1835859,1480,8.0E-4
Night,1589695,418,3.0E-4


In [0]:
fraud_monthly_summary_gold = (
    transactions_enriched
    .groupBy("transaction_year", "transaction_month")
    .agg(
        count("*").alias("total_transaction_count"),
        spark_sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transaction_count"),
        spark_round(spark_sum(when(col("is_fraud") == True, 1).otherwise(0)) / count("*"), 4).alias("fraud_rate")
    )
    .orderBy("transaction_year", "transaction_month")
)

display(fraud_monthly_summary_gold.limit(20))

transaction_year,transaction_month,total_transaction_count,fraud_transaction_count,fraud_rate
2010,1,101209,107,0.0011
2010,2,93470,259,0.0028
2010,3,103345,261,0.0025
2010,4,100169,237,0.0024
2010,5,104773,274,0.0026
2010,6,102677,182,0.0018
2010,7,106034,244,0.0023
2010,8,107547,229,0.0021
2010,9,103902,193,0.0019
2010,10,106150,224,0.0021


In [0]:
from pyspark.sql.functions import count, sum as spark_sum, when, col, round as spark_round

fraud_merchant_gold = (
    transactions_enriched
    .withColumn(
        "amount_bucket",
        when(col("abs_amount") < 50, "low_value")
        .when((col("abs_amount") >= 50) & (col("abs_amount") < 200), "medium_value")
        .when((col("abs_amount") >= 200) & (col("abs_amount") < 500), "high_value")
        .otherwise("very_high_value")
    )
    .groupBy("mcc_description", "amount_bucket")
    .agg(
        count("*").alias("total_transaction_count"),
        spark_sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transaction_count"),
        spark_round(spark_sum(when(col("is_fraud") == True, col("abs_amount")).otherwise(0.0)), 2).alias("total_fraud_amount"),
        spark_round(spark_sum(when(col("is_fraud") == True, 1).otherwise(0)) / count("*"), 4).alias("fraud_rate")
    )
    .orderBy(col("total_fraud_amount").desc())
)

display(fraud_merchant_gold.limit(20))

mcc_description,amount_bucket,total_transaction_count,fraud_transaction_count,total_fraud_amount,fraud_rate
Cruise Lines,very_high_value,362,121,176298.13,0.3343
Department Stores,medium_value,208766,1043,108927.87,0.005
Wholesale Clubs,medium_value,300442,529,60183.68,0.0018
Department Stores,high_value,5712,205,59464.09,0.0359
Money Transfer,medium_value,485580,504,53922.06,0.001
Discount Stores,medium_value,68645,431,47198.58,0.0063
Wholesale Clubs,high_value,8677,141,40171.04,0.0162
Department Stores,very_high_value,692,54,39211.72,0.078
Passenger Railways,high_value,11125,102,33506.01,0.0092
Gardening Supplies,high_value,10096,92,31353.78,0.0091


In [0]:
from pyspark.sql.functions import count, sum as spark_sum, when, col, round as spark_round

fraud_by_specific_merchant_gold = (
    transactions_enriched
    .groupBy("merchant_id", "merchant_city", "merchant_state", "mcc_description")
    .agg(
        count("*").alias("total_transaction_count"),
        spark_sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transaction_count"),
        spark_round(spark_sum(when(col("is_fraud") == True, 1).otherwise(0)) / count("*"), 4).alias("fraud_rate")
    )
    .filter(col("fraud_transaction_count") > 0)
    .orderBy(col("fraud_transaction_count").desc())
)

display(fraud_by_specific_merchant_gold.limit(20))

merchant_id,merchant_city,merchant_state,mcc_description,total_transaction_count,fraud_transaction_count,fraud_rate
60569,ONLINE,null,Wholesale Clubs,1104,759,0.6875
27092,ONLINE,null,Money Transfer,1055,715,0.6777
76639,ONLINE,null,Electronics Stores,397,284,0.7154
32858,ONLINE,null,Department Stores,431,283,0.6566
83018,Rome,Italy,Discount Stores,391,236,0.6036
48919,Rome,Italy,Department Stores,332,221,0.6657
99370,Rome,Italy,Department Stores,326,195,0.5982
47399,ONLINE,null,"Digital Goods - Media, Books, Apps",20858,176,0.0084
34490,ONLINE,null,Miscellaneous Home Furnishing Stores,230,160,0.6957
88260,Rome,Italy,"Grocery Stores, Supermarkets",390,149,0.3821


In [0]:
from pyspark.sql.functions import min as spark_min

first_fraud_date = (
    transactions_enriched
    .filter(col("is_fraud") == True)
    .groupBy("client_id")
    .agg(spark_min("date").alias("first_fraud_date"))
)

display(first_fraud_date.limit(10))

client_id,first_fraud_date
1540,2012-03-08T15:03:00.000Z
1094,2015-08-02T11:01:00.000Z
1276,2016-11-20T12:04:00.000Z
619,2016-03-17T12:32:00.000Z
402,2019-06-21T18:10:00.000Z
801,2010-09-12T07:01:00.000Z
1629,2018-09-19T13:16:00.000Z
165,2016-03-13T10:06:00.000Z
1226,2010-11-07T02:27:00.000Z
1492,2012-05-10T10:18:00.000Z


In [0]:
from pyspark.sql.functions import count, sum as spark_sum, when, col

fraud_user_gold = (
    transactions_enriched.alias("t")
    .join(users.select("client_id", "gender", "current_age", "credit_score", "yearly_income", "total_debt").alias("u"), on="client_id", how="left")
    .groupBy("client_id", "gender", "current_age", "credit_score", "yearly_income", "total_debt")
    .agg(
        count("*").alias("total_transaction_count"),
        spark_sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transaction_count")
    )
    .orderBy(col("fraud_transaction_count").desc())
)

display(fraud_user_gold.limit(20))

client_id,gender,current_age,credit_score,yearly_income,total_debt,total_transaction_count,fraud_transaction_count
1102,Female,49,707,94733.0,0.0,17775,58
209,Female,61,716,29206.0,25966.0,12899,52
27,Female,78,613,23821.0,22427.0,9784,45
155,Female,86,688,9678.0,812.0,8397,44
1128,Male,47,680,34606.0,23909.0,8268,43
1851,Male,48,727,75682.0,37163.0,6146,42
989,Male,78,727,113514.0,16524.0,12069,42
1741,Male,92,707,24960.0,889.0,8626,42
1649,Male,64,729,399.0,323.0,2336,41
1416,Female,73,709,48750.0,18724.0,10310,39


In [0]:
from pyspark.sql.functions import when, col, lit, avg, count, round as spark_round

user_behavior_change = (
    transactions_enriched.alias("t")
    .join(first_fraud_date.alias("f"), on="client_id", how="inner")
    .withColumn(
        "period",
        when(col("t.date") < col("f.first_fraud_date"), "before_fraud")
        .when(col("t.date") > col("f.first_fraud_date"), "after_fraud")
        .otherwise("first_fraud_day")
    )
    .groupBy("client_id", "period")
    .agg(
        count("*").alias("transaction_count"),
        spark_round(avg("abs_amount"), 2).alias("avg_amount")
    )
    .orderBy("client_id", "period")
)

display(user_behavior_change.limit(20))

client_id,period,transaction_count,avg_amount
0,after_fraud,5246,61.72
0,before_fraud,7548,60.56
0,first_fraud_day,1,5.13
1,after_fraud,2817,39.64
1,before_fraud,7255,35.32
1,first_fraud_day,1,2.20
2,after_fraud,3432,33.89
2,before_fraud,7179,35.62
2,first_fraud_day,1,298.30
3,after_fraud,3865,50.83


In [0]:
fraud_daily_summary_gold.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("jarvis_databricks.gold.fraud_daily_summary_gold")

fraud_by_time_of_day_gold.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("jarvis_databricks.gold.fraud_by_time_of_day_gold")

fraud_monthly_summary_gold.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("jarvis_databricks.gold.fraud_monthly_summary_gold")

fraud_merchant_gold.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("jarvis_databricks.gold.fraud_merchant_gold")

fraud_by_specific_merchant_gold.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("jarvis_databricks.gold.fraud_by_specific_merchant_gold")

fraud_user_gold.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("jarvis_databricks.gold.fraud_user_gold")

user_behavior_change.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("jarvis_databricks.gold.user_behavior_change_gold")

In [0]:
%sql
SHOW TABLES IN jarvis_databricks.gold;

database,tableName,isTemporary
gold,fraud_by_specific_merchant_gold,false
gold,fraud_by_time_of_day_gold,false
gold,fraud_daily_summary_gold,false
gold,fraud_merchant_gold,false
gold,fraud_monthly_summary_gold,false
gold,fraud_user_gold,false
gold,user_behavior_change_gold,false
